# RF-DETR 74-Class Hidden45 Training Notebook

## Goal

Train RF-DETR on the 74-class pill dataset built from:

- base 56 classes, filled to at least 45 annotations across train+valid
- hidden N57-N74 crop classes
- original downloaded competition test images for inference-only `test` split

The notebook is a runnable handoff wrapper around the scripts in this folder. The default config is tuned for local Apple Silicon MPS. By default it prepares and validates the dataset, then runs a dry-run. Set `RUN_FULL_TRAIN = True` in the parameter cell to start real training.

## Setup

Use this notebook from the team repo. If opened from the repo root, it will move into `RF_DETR_split_ver` automatically.

Key assumptions:

- `rfdetr[train,loggers]` is installed in the active Python kernel.
- The prepared 56-class and hidden N18 source folders exist locally or are restored from Drive.
- Test images come from the original downloaded competition `test_images` folder and have no public annotations.
- Local MPS defaults use `batch_size=1`, `grad_accum_steps=16`, `num_workers=2`, `pin_memory=false`, and `multi_scale=false` for stable memory use.

In [1]:
# Parameters
from pathlib import Path

RUN_INSTALL_DEPS = False      # Set True only in a fresh Colab/kernel.
RUN_PREPARE_DATASET = True    # Rebuild RF-DETR train/valid/test folder.
RUN_DRY_RUN = True            # Validate config without launching long training.
RUN_FULL_TRAIN = False        # Change to True when ready to train.
TRAIN_EPOCHS_OVERRIDE = None  # Example: 3 for a short local run, or None to use config_74_hidden45.yaml.

# Local defaults used by the current project workspace.
BASE56_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco')
HIDDEN18_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import')
TEST_IMAGES_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images')
RFDETR_DATASET_DIR = Path('/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45')
CONFIG_PATH = Path('config_74_hidden45.yaml')

LINK_MODE = 'symlink'  # symlink avoids duplicate image copies. Use copy for portable upload folders.
DEVICE_NOTE = 'Local MPS defaults are active. For Colab/GPU, set train.device=cuda and train.pin_memory=true.'
print(DEVICE_NOTE)

Local MPS defaults are active. For Colab/GPU, set train.device=cuda and train.pin_memory=true.


In [2]:
# Imports and working directory
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

start_cwd = Path.cwd()
if (start_cwd / 'RF_DETR_split_ver').exists():
    os.chdir(start_cwd / 'RF_DETR_split_ver')
elif not (start_cwd / 'prepare_74_hidden45_dataset.py').exists():
    raise RuntimeError(f'Open this notebook from repo root or RF_DETR_split_ver, not {start_cwd}')

WORK_DIR = Path.cwd()
print('WORK_DIR:', WORK_DIR)
print('Python:', sys.executable)
print('Config:', (WORK_DIR / CONFIG_PATH).resolve())

WORK_DIR: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver
Python: /Users/pio/.DataAnalysis/bin/python3.11
Config: /Users/pio/Documents/AIENGINEERCOURSE/ai12-team01-rfdetr/RF_DETR_split_ver/config_74_hidden45.yaml


In [3]:
# Optional dependency install
if RUN_INSTALL_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK_DIR.parent / 'requirements.txt')], check=True)
else:
    print('Skipping dependency install. Set RUN_INSTALL_DEPS=True in a fresh environment.')

Skipping dependency install. Set RUN_INSTALL_DEPS=True in a fresh environment.


In [4]:
# Environment check
import torch

try:
    import rfdetr
    rfdetr_status = 'ok'
except Exception as exc:
    rfdetr_status = f'IMPORT FAILED: {exc}'

env_summary = {
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'mps_available': bool(getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()),
    'rfdetr': rfdetr_status,
}
print(json.dumps(env_summary, indent=2, ensure_ascii=False))

/Users/pio/.DataAnalysis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "torch_version": "2.12.0",
  "cuda_available": false,
  "mps_available": true,
  "rfdetr": "ok"
}


/Users/pio/.DataAnalysis/lib/python3.11/site-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


## Data

This section validates source folders and prepares the RF-DETR directory layout:

```text
rfdetr_dataset_74_hidden45/
  train/_annotations.coco.json
  valid/_annotations.coco.json
  test/_annotations.coco.json
```

`train` and `valid` contain labeled images. `test` contains the original downloaded test images with an empty annotation list.

In [5]:
# Validate source paths
paths = {
    'BASE56_DIR': BASE56_DIR,
    'HIDDEN18_DIR': HIDDEN18_DIR,
    'TEST_IMAGES_DIR': TEST_IMAGES_DIR,
}
missing = {name: str(path) for name, path in paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(json.dumps(missing, indent=2, ensure_ascii=False))

for name, path in paths.items():
    print(f'{name}: {path}')

BASE56_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco
HIDDEN18_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import
TEST_IMAGES_DIR: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images


In [6]:
# Prepare RF-DETR dataset
if RUN_PREPARE_DATASET:
    prepare_cmd = [
        sys.executable,
        'prepare_74_hidden45_dataset.py',
        '--base56-dir', str(BASE56_DIR),
        '--hidden18-dir', str(HIDDEN18_DIR),
        '--test-images-dir', str(TEST_IMAGES_DIR),
        '--out-dir', str(RFDETR_DATASET_DIR),
        '--link-mode', LINK_MODE,
    ]
    print('run:', ' '.join(prepare_cmd))
    subprocess.run(prepare_cmd, check=True)
else:
    print('Skipping dataset preparation. Using existing:', RFDETR_DATASET_DIR)

run: /Users/pio/.DataAnalysis/bin/python3.11 prepare_74_hidden45_dataset.py --base56-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco --hidden18-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import --test-images-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images --out-dir /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45 --link-mode symlink


{
  "base56_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco",
  "hidden18_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import",
  "test_images_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images",
  "out_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45",
  "link_mode": "symlink",
  "class_count": 74,
  "category_id_semantics": "COCO category_id is the numeric K-code. RF-DETR remaps sparse ids internally.",
  "splits": [
    {
      "split": "train",
      "source_split": "train",
      "images": 3502,
      "annotations": 3965,
      "categories": 74,
      "base56_images": 1875,
      "hidden18_images": 1627
    },
    {
      "split": "valid",
      "source_split": "val",
      "images": 376,
      "annotations": 452,
      "categories": 74,
      "base56_images": 218,
      "hid

In [7]:
# Load dataset summary
summary_path = RFDETR_DATASET_DIR / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(pd.DataFrame(summary['splits']))
print('category_id_semantics:', summary['category_id_semantics'])

,split,source_split,images,annotations,categories,base56_images,hidden18_images,test_images_dir
0,train,train,3502,3965,74,1875.0,1627.0,NaN
1,valid,val,376,452,74,218.0,158.0,NaN
2,test,original_test,842,0,74,NaN,NaN,/Users/pio/Documents/AIENGINEERCOURSE/detectio...


category_id_semantics: COCO category_id is the numeric K-code. RF-DETR remaps sparse ids internally.


In [8]:
# Sanity checks: split sizes, K-code ids, bbox bounds, and 45-fill coverage
def load_split(split: str) -> dict:
    return json.loads((RFDETR_DATASET_DIR / split / '_annotations.coco.json').read_text(encoding='utf-8'))

splits = {split: load_split(split) for split in ['train', 'valid', 'test']}
rows = []
for split, data in splits.items():
    category_ids = sorted(int(category['id']) for category in data['categories'])
    annotation_ids = sorted({int(annotation['category_id']) for annotation in data['annotations']})
    file_count = sum(1 for path in (RFDETR_DATASET_DIR / split).iterdir() if path.name != '_annotations.coco.json')
    rows.append({
        'split': split,
        'json_images': len(data['images']),
        'files': file_count,
        'annotations': len(data['annotations']),
        'categories': len(category_ids),
        'category_id_min': min(category_ids),
        'category_id_max': max(category_ids),
        'annotation_id_min': min(annotation_ids) if annotation_ids else None,
        'annotation_id_max': max(annotation_ids) if annotation_ids else None,
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

assert summary_df.loc[summary_df['split'].eq('train'), 'categories'].item() == 74
assert summary_df.loc[summary_df['split'].eq('valid'), 'categories'].item() == 74
assert summary_df.loc[summary_df['split'].eq('test'), 'json_images'].item() == 842
assert summary_df.loc[summary_df['split'].eq('test'), 'annotations'].item() == 0

# Validate bbox bounds on labeled splits.
for split in ['train', 'valid']:
    data = splits[split]
    images = {image['id']: image for image in data['images']}
    for annotation in data['annotations']:
        image = images[annotation['image_id']]
        x, y, w, h = map(float, annotation['bbox'])
        assert w > 0 and h > 0, (split, annotation)
        assert x >= 0 and y >= 0, (split, image['file_name'], annotation['bbox'])
        assert x + w <= image['width'] + 1e-6, (split, image['file_name'], annotation['bbox'])
        assert y + h <= image['height'] + 1e-6, (split, image['file_name'], annotation['bbox'])

train_counts = Counter(int(annotation['category_id']) for annotation in splits['train']['annotations'])
valid_counts = Counter(int(annotation['category_id']) for annotation in splits['valid']['annotations'])
combined_counts = train_counts + valid_counts
all_category_ids = sorted(int(category['id']) for category in splits['train']['categories'])
min_combined = min(combined_counts[category_id] for category_id in all_category_ids)
classes_below_45 = [category_id for category_id in all_category_ids if combined_counts[category_id] < 45]
print('train+valid min annotations per class:', min_combined)
print('classes below 45:', classes_below_45)
assert not classes_below_45

category_mapping = pd.read_csv(RFDETR_DATASET_DIR / 'category_mapping.csv')
display(category_mapping.head())
display(category_mapping.tail())

,split,json_images,files,annotations,categories,category_id_min,category_id_max,annotation_id_min,annotation_id_max
0,train,3502,3502,3965,74,1900,44199,1900.0,44199.0
1,valid,376,376,452,74,1900,44199,1900.0,44199.0
2,test,842,842,0,74,1900,44199,NaN,NaN


train+valid min annotations per class: 45
classes below 45: []


,category_id,rfdetr_internal_label,n_number,source_dataset,source_category_id,name,mapping_code,print_front,print_back,candidate_status,decision_note
0,1900,0,N01,base56_45fill,0,보령부스파정 5mg,K-001900,BSP,5,NaN,NaN
1,2483,1,N02,base56_45fill,1,뮤테란캡슐 100mg,K-002483,Hanwha MUC100,NaN,NaN,NaN
2,3351,2,N03,base56_45fill,2,일양하이트린정 2mg,K-003351,I분할선Y,HT,NaN,NaN
3,3483,3,N04,base56_45fill,3,기넥신에프정(은행엽엑스)(수출용),K-003483,SK,G40,NaN,NaN
4,3544,4,N05,base56_45fill,4,무코스타정(레바미피드)(비매품),K-003544,OG33,NaN,NaN,NaN


,category_id,rfdetr_internal_label,n_number,source_dataset,source_category_id,name,mapping_code,print_front,print_back,candidate_status,decision_note
69,35206,69,N53,base56_45fill,52,아토젯정 10/40mg,K-035206,337,NaN,NaN,NaN
70,36637,70,N54,base56_45fill,53,로수젯정10/5밀리그램,K-036637,R5,분할선,NaN,NaN
71,38162,71,N55,base56_45fill,54,로수바미브정 10/20mg,K-038162,RE20,Y분할선H,NaN,NaN
72,41768,72,N56,base56_45fill,55,카발린캡슐 25mg,K-041768,CJ PGN 25,NaN,NaN,NaN
73,44199,73,N66,hidden18_45fill,44199,케이캡정 50mg,K-044199,K분할선50,분할선,hidden_confirmed,K|50 반복 unknown과 케이캡정 50mg exact match. K|150처...


## Training

Run a dry-run first. Then set `RUN_FULL_TRAIN = True` in the parameter cell and rerun the final training cell.

Local MPS baseline in `config_74_hidden45.yaml`:

```yaml
train:
  device: mps
  batch_size: 1
  grad_accum_steps: 16
  num_workers: 2
  pin_memory: false
  multi_scale: false
```

For Colab/CUDA, update before training:

```yaml
train:
  device: cuda
  pin_memory: true
  num_workers: 2  # raise to 4 only if the runtime and Drive I/O are stable
output:
  local_output_dir: /content/drive/MyDrive/.../rfdetr_outputs
  backup_dir: /content/drive/MyDrive/.../rfdetr_outputs/best
```

In [9]:
# Inspect training config
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

data:
  dataset_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45
  base56_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/train_56_45_merged_coco
  hidden18_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/aihub_prepared/hidden_train_import
  test_images_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/sprint_ai_project1_data/test_images
model:
  variant: nano
  tag: nano_74_hidden45_mps
train:
  epochs: 12
  batch_size: 1
  grad_accum_steps: 16
  seed: 42
  lr: 0.0001
  lr_encoder: 0.00015
  weight_decay: 0.0001
  lr_scheduler: cosine
  warmup_epochs: 1.0
  lr_min_factor: 0.05
  checkpoint_interval: 1
  eval_interval: 1
  early_stopping: true
  early_stopping_patience: 10
  early_stopping_min_delta: 0.001
  early_stopping_use_ema: true
  use_ema: true
  tensorboard: true
  progress_bar: tqdm
  log_per_class_metrics: true
  run_test: false
  eval_max_dets: 500
  num_workers: 2

In [10]:
# Dry-run training entrypoint
if RUN_DRY_RUN:
    dry_run_cmd = [sys.executable, 'train_74_hidden45.py', '--config', str(CONFIG_PATH), '--dry-run']
    print('run:', ' '.join(dry_run_cmd))
    subprocess.run(dry_run_cmd, check=True)
else:
    print('Skipping dry-run.')

run: /Users/pio/.DataAnalysis/bin/python3.11 train_74_hidden45.py --config config_74_hidden45.yaml --dry-run


/Users/pio/.DataAnalysis/lib/python3.11/site-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


{
  "dataset_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_dataset_74_hidden45",
  "output_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps",
  "backup_dir": "/Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/best",
  "model_variant": "nano",
  "model_tag": "nano_74_hidden45_mps",
  "train_kwargs": {
    "epochs": 12,
    "batch_size": 1,
    "grad_accum_steps": 16,
    "seed": 42,
    "lr": 0.0001,
    "lr_encoder": 0.00015,
    "weight_decay": 0.0001,
    "lr_scheduler": "cosine",
    "warmup_epochs": 1.0,
    "lr_min_factor": 0.05,
    "checkpoint_interval": 1,
    "eval_interval": 1,
    "early_stopping": true,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001,
    "early_stopping_use_ema": true,
    "use_ema": true,
    "tensorboard": true,
    "progress_bar": "tqdm",
    "log_per_class_metrics": true,
    "run_test": false,
    "eval_max_dets"

In [11]:
# Full training
train_cmd = [sys.executable, 'train_74_hidden45.py', '--config', str(CONFIG_PATH)]
if TRAIN_EPOCHS_OVERRIDE is not None:
    train_cmd += ['--epochs', str(TRAIN_EPOCHS_OVERRIDE)]

print('full train command:', ' '.join(train_cmd))
if RUN_FULL_TRAIN:
    env = os.environ.copy()
    env.setdefault('WANDB_DISABLED', 'true')
    env.setdefault('WANDB_MODE', 'disabled')
    subprocess.run(train_cmd, check=True, env=env)
else:
    print('RUN_FULL_TRAIN=False, so training was not launched.')
    print('Set RUN_FULL_TRAIN=True and rerun this cell to train.')

full train command: /Users/pio/.DataAnalysis/bin/python3.11 train_74_hidden45.py --config config_74_hidden45.yaml
RUN_FULL_TRAIN=False, so training was not launched.
Set RUN_FULL_TRAIN=True and rerun this cell to train.


In [12]:
# Locate outputs/checkpoints
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
model_tag = config['model'].get('tag', config['model']['variant'])
output_dir = Path(config['output']['local_output_dir']) / model_tag
backup_dir = Path(config['output']['backup_dir'])

print('output_dir:', output_dir)
print('backup_dir:', backup_dir)
if output_dir.exists():
    for path in sorted(output_dir.glob('*')):
        print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB' if path.is_file() else '<dir>')
else:
    print('No output_dir yet. Run full training first.')

if backup_dir.exists():
    best_candidates = sorted(backup_dir.glob(f'{model_tag}*'))
    print('best candidates:')
    for path in best_candidates:
        print('-', path, f'{path.stat().st_size / (1024 * 1024):.1f} MB')
else:
    print('No backup_dir yet.')

output_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/nano_74_hidden45_mps
backup_dir: /Users/pio/Documents/AIENGINEERCOURSE/detectionproject/working/rfdetr_outputs/best
run_summary.json 0.0 MB
best candidates:


## Checks

Expected safe-run result with default parameters:

- dataset preparation succeeds
- `train` has 3502 images and 3965 annotations
- `valid` has 376 images and 452 annotations
- `test` has 842 original test images and 0 annotations
- train+valid has no class below 45 annotations
- dry-run exits successfully

When `RUN_FULL_TRAIN=True`, the best checkpoint is copied to the configured `backup_dir` after training if `checkpoint_best_total.pth` is produced.